In [1]:
import numpy as np
import pandas as pd

from linearmodels import PooledOLS          # Pooled model
from linearmodels import RandomEffects      # Random-effect model
from linearmodels import PanelOLS           # Fixed-effect model
from linearmodels import FirstDifferenceOLS # First difference model

from linearmodels.panel import compare      # Compare the results of multiple models
from statsmodels.api import add_constant    # for matrices of regression design

# 1.2 Модель 3. Результаты оценивания

In [2]:
df = pd.read_csv('Wages.csv')
df.head()

,exp,wks,bluecol,ind,south,smsa,married,sex,union,ed,black,lwage,id,time
0,3,32,no,0,yes,no,yes,male,no,9,no,5.56068,1,1
1,4,43,no,0,yes,no,yes,male,no,9,no,5.72031,1,2
2,5,40,no,0,yes,no,yes,male,no,9,no,5.99645,1,3
3,6,39,no,0,yes,no,yes,male,no,9,no,5.99645,1,4
4,7,42,no,1,yes,no,yes,male,no,9,no,6.06146,1,5


In [3]:
panel_df = df.set_index(['id', 'time'])
panel_df.head()

exp  wks bluecol  ind south smsa married   sex union  ed black  \
id time                                                                   
1  1       3   32      no    0   yes   no     yes  male    no   9    no   
   2       4   43      no    0   yes   no     yes  male    no   9    no   
   3       5   40      no    0   yes   no     yes  male    no   9    no   
   4       6   39      no    0   yes   no     yes  male    no   9    no   
   5       7   42      no    1   yes   no     yes  male    no   9    no   

           lwage  
id time           
1  1     5.56068  
   2     5.72031  
   3     5.99645  
   4     5.99645  
   5     6.06146

In [4]:
panel_df['d_lwage'] = panel_df['lwage'].groupby(level=0).diff()
panel_df.head(20)

exp  wks bluecol  ind south smsa married   sex union  ed black  \
id time                                                                   
1  1       3   32      no    0   yes   no     yes  male    no   9    no   
   2       4   43      no    0   yes   no     yes  male    no   9    no   
   3       5   40      no    0   yes   no     yes  male    no   9    no   
   4       6   39      no    0   yes   no     yes  male    no   9    no   
   5       7   42      no    1   yes   no     yes  male    no   9    no   
   6       8   35      no    1   yes   no     yes  male    no   9    no   
   7       9   32      no    1   yes   no     yes  male    no   9    no   
2  1      30   34     yes    0    no   no     yes  male    no  11    no   
   2      31   27     yes    0    no   no     yes  male    no  11    no   
   3      32   33     yes    1    no   no     yes  male   yes  11    no   
   4      33   30     yes    1    no   no     yes  male    no  11    no   
   5      34   30     yes    1    no   no     yes  male    no  11    no   
   6      35   37     yes    1    no   no     yes  male    no  11    no   
   7      36   30     yes    1    no   no     yes  male    no  11    no   
3  1       6   50     yes    1    no   no     yes  male   yes  12    no   
   2       7   51     yes    1    no   no     yes  male   yes  12    no   
   3       8   50     yes    1    no   no     yes  male   yes  12    no   
   4       9   52     yes    1    no   no     yes  male   yes  12    no   
   5      10   52     yes    1    no   no     yes  male   yes  12    no   
   6      11   52     yes    1    no   no      no  male   yes  12    no   

           lwage  d_lwage  
id time                    
1  1     5.56068      NaN  
   2     5.72031  0.15963  
   3     5.99645  0.27614  
   4     5.99645  0.00000  
   5     6.06146  0.06501  
   6     6.17379  0.11233  
   7     6.24417  0.07038  
2  1     6.16331      NaN  
   2     6.21461  0.05130  
   3     6.26340  0.04879  
   4     6.54391  0.28051  
   5     6.69703  0.15312  
   6     6.79122  0.09419  
   7     6.81564  0.02442  
3  1     5.65249      NaN  
   2     6.43615  0.78366  
   3     6.54822  0.11207  
   4     6.60259  0.05437  
   5     6.69580  0.09321  
   6     6.77878  0.08298

In [5]:
mod_pl = PooledOLS.from_formula(formula='d_lwage~1+ed+exp+I(exp**2)+wks+south+smsa+married+union+bluecol', data=panel_df)
res_pl = mod_pl.fit()
res_pl.params.round(4)

D:\Anaconda\Lib\site-packages\linearmodels\panel\model.py:919: MissingValueWarning: 
Inputs contain missing values. Dropping rows with missing observations.
  super().__init__(dependent, exog, weights=weights, check_rank=check_rank)


Intercept         0.0998
ed                0.0014
exp              -0.0027
I(exp ** 2)       0.0000
wks               0.0003
south[T.yes]      0.0018
smsa[T.yes]      -0.0011
married[T.yes]    0.0036
union[T.yes]      0.0020
bluecol[T.yes]   -0.0073
Name: parameter, dtype: float64

In [6]:
mod_re = RandomEffects.from_formula(formula='d_lwage~1+ed+exp+I(exp**2)+wks+south+smsa+married+union+bluecol', data=panel_df)
res_re = mod_re.fit()
res_re.params.round(4)

D:\Anaconda\Lib\site-packages\linearmodels\panel\model.py:2751: MissingValueWarning: 
Inputs contain missing values. Dropping rows with missing observations.
  super().__init__(dependent, exog, weights=weights, check_rank=check_rank)


Intercept         0.0998
ed                0.0014
exp              -0.0027
I(exp ** 2)       0.0000
wks               0.0003
south[T.yes]      0.0018
smsa[T.yes]      -0.0011
married[T.yes]    0.0036
union[T.yes]      0.0020
bluecol[T.yes]   -0.0073
Name: parameter, dtype: float64

In [7]:
mod_fe = PanelOLS.from_formula(formula='d_lwage~1+ed+exp+I(exp**2)+wks+south+smsa+married+union+bluecol+EntityEffects', data=panel_df, drop_absorbed=True)
res_fe = mod_fe.fit()
res_fe.params.round(4)

D:\Anaconda\Lib\site-packages\linearmodels\panel\model.py:1258: MissingValueWarning: 
Inputs contain missing values. Dropping rows with missing observations.
  super().__init__(dependent, exog, weights=weights, check_rank=check_rank)
C:\Temp\ipykernel_6528\1847005776.py:2: AbsorbingEffectWarning: 
Variables have been fully absorbed and have removed from the regression:

ed

  res_fe = mod_fe.fit()


Intercept         0.2683
exp              -0.0106
I(exp ** 2)       0.0001
wks              -0.0001
south[T.yes]      0.0704
smsa[T.yes]       0.0020
married[T.yes]   -0.0279
union[T.yes]      0.0170
bluecol[T.yes]   -0.0465
Name: parameter, dtype: float64

# 3.3 Модель 3. Результаты оценивания

In [8]:
df = pd.read_csv('EmplUK.csv')
df.head()

,firm,year,sector,emp,wage,capital,output
0,1,1977,7,5.041,13.1516,0.5894,95.707199
1,1,1978,7,5.600,12.3018,0.6318,97.356903
2,1,1979,7,5.015,12.8395,0.6771,99.608299
3,1,1980,7,4.715,13.8039,0.6171,100.550100
4,1,1981,7,4.093,14.2897,0.5076,99.558098


In [9]:
panel_df = df.set_index(['firm', 'year'])
panel_df.head(20)

sector        emp       wage    capital      output
firm year                                                     
1    1977       7   5.041000  13.151600   0.589400   95.707199
     1978       7   5.600000  12.301800   0.631800   97.356903
     1979       7   5.015000  12.839500   0.677100   99.608299
     1980       7   4.715000  13.803900   0.617100  100.550100
     1981       7   4.093000  14.289700   0.507600   99.558098
     1982       7   3.166000  14.868100   0.422900   98.615097
     1983       7   2.936000  13.778400   0.392000  100.030100
2    1977       7  71.319000  14.790900  16.936300   95.707199
     1978       7  70.642998  14.103600  17.242201   97.356903
     1979       7  70.917999  14.953400  17.541300   99.608299
     1980       7  72.030998  15.491000  17.657400  100.550100
     1981       7  73.689003  16.196899  16.713301   99.558098
     1982       7  72.418999  16.131399  16.246901   98.615097
     1983       7  68.517998  16.305099  17.369600  100.030100
3    1977       7  19.156000  22.691999   7.097500   95.707199
     1978       7  19.440001  20.693800   6.946900   97.356903
     1979       7  19.900000  21.204800   6.856500   99.608299
     1980       7  20.240000  22.197001   6.654700  100.550100
     1981       7  19.570000  24.871401   6.213600   99.558098
     1982       7  18.125000  24.844700   5.714600   98.615097

In [10]:
var_names=np.array(['wage', 'capital', 'output'])

In [11]:
panel_df['lag_log_'+var_names] = np.log(panel_df[var_names]).groupby(level=0).shift()
panel_df.head(20)

sector        emp       wage    capital      output  lag_log_wage  \
firm year                                                                      
1    1977       7   5.041000  13.151600   0.589400   95.707199           NaN   
     1978       7   5.600000  12.301800   0.631800   97.356903      2.576543   
     1979       7   5.015000  12.839500   0.677100   99.608299      2.509746   
     1980       7   4.715000  13.803900   0.617100  100.550100      2.552526   
     1981       7   4.093000  14.289700   0.507600   99.558098      2.624951   
     1982       7   3.166000  14.868100   0.422900   98.615097      2.659539   
     1983       7   2.936000  13.778400   0.392000  100.030100      2.699218   
2    1977       7  71.319000  14.790900  16.936300   95.707199           NaN   
     1978       7  70.642998  14.103600  17.242201   97.356903      2.694012   
     1979       7  70.917999  14.953400  17.541300   99.608299      2.646430   
     1980       7  72.030998  15.491000  17.657400  100.550100      2.704939   
     1981       7  73.689003  16.196899  16.713301   99.558098      2.740259   
     1982       7  72.418999  16.131399  16.246901   98.615097      2.784820   
     1983       7  68.517998  16.305099  17.369600  100.030100      2.780768   
3    1977       7  19.156000  22.691999   7.097500   95.707199           NaN   
     1978       7  19.440001  20.693800   6.946900   97.356903      3.122012   
     1979       7  19.900000  21.204800   6.856500   99.608299      3.029834   
     1980       7  20.240000  22.197001   6.654700  100.550100      3.054228   
     1981       7  19.570000  24.871401   6.213600   99.558098      3.099957   
     1982       7  18.125000  24.844700   5.714600   98.615097      3.213719   

           lag_log_capital  lag_log_output  
firm year                                   
1    1977              NaN             NaN  
     1978        -0.528650        4.561294  
     1979        -0.459182        4.578384  
     1980        -0.389936        4.601245  
     1981        -0.482724        4.610656  
     1982        -0.678062        4.600741  
     1983        -0.860620        4.591224  
2    1977              NaN             NaN  
     1978         2.829459        4.561294  
     1979         2.847360        4.578384  
     1980         2.864558        4.601245  
     1981         2.871155        4.610656  
     1982         2.816205        4.600741  
     1983         2.787902        4.591224  
3    1977              NaN             NaN  
     1978         1.959743        4.561294  
     1979         1.938296        4.578384  
     1980         1.925197        4.601245  
     1981         1.895323        4.610656  
     1982         1.826740        4.600741

In [12]:
mod_pl = PooledOLS.from_formula(formula='np.log(emp)~1+np.log(wage)+lag_log_wage+np.log(capital)+lag_log_capital+np.log(output)+lag_log_output', data=panel_df)
res_pl = mod_pl.fit()
res_pl.params.round(4)

D:\Anaconda\Lib\site-packages\linearmodels\panel\model.py:919: MissingValueWarning: 
Inputs contain missing values. Dropping rows with missing observations.
  super().__init__(dependent, exog, weights=weights, check_rank=check_rank)


Intercept         -0.5427
np.log(wage)      -0.3561
lag_log_wage      -0.0705
np.log(capital)    0.3297
lag_log_capital    0.4840
np.log(output)    -0.2797
lag_log_output     0.9812
Name: parameter, dtype: float64

In [13]:
mod_re = RandomEffects.from_formula(formula='np.log(emp)~1+np.log(wage)+lag_log_wage+np.log(capital)+lag_log_capital+np.log(output)+lag_log_output', data=panel_df)
res_re = mod_re.fit()
res_re.params.round(4)

D:\Anaconda\Lib\site-packages\linearmodels\panel\model.py:2751: MissingValueWarning: 
Inputs contain missing values. Dropping rows with missing observations.
  super().__init__(dependent, exog, weights=weights, check_rank=check_rank)


Intercept          1.4235
np.log(wage)      -0.5499
lag_log_wage       0.0921
np.log(capital)    0.5569
lag_log_capital    0.1392
np.log(output)     0.5719
lag_log_output    -0.2730
Name: parameter, dtype: float64

In [14]:
mod_fe = PanelOLS.from_formula(formula='np.log(emp)~1+np.log(wage)+lag_log_wage+np.log(capital)+lag_log_capital+np.log(output)+lag_log_output+EntityEffects', data=panel_df, drop_absorbed=True)
res_fe = mod_fe.fit()
res_fe.params.round(4)

D:\Anaconda\Lib\site-packages\linearmodels\panel\model.py:1258: MissingValueWarning: 
Inputs contain missing values. Dropping rows with missing observations.
  super().__init__(dependent, exog, weights=weights, check_rank=check_rank)


Intercept          0.8424
np.log(wage)      -0.5494
lag_log_wage       0.0632
np.log(capital)    0.4956
lag_log_capital    0.0857
np.log(output)     0.5751
lag_log_output    -0.1464
Name: parameter, dtype: float64